# 60 — Finalizar ingestão
Consolida os resultados das etapas e atualiza o estado definitivo de cada vídeo.

In [ ]:
import json

from youtube_etl_genai.main import _get_spark_session
from youtube_etl_genai.observability import TaskExecution, configure_job_logging
from youtube_etl_genai.pipeline import finalize_ingestion_step

TASK_KEY = "finalize_ingestion"
configure_job_logging()

for name, default in [("ingestion_id", ""), ("catalog", "youtube_lakehouse"), ("task_run_id", "")]:
    dbutils.widgets.text(name, default)

spark = _get_spark_session()
catalog = dbutils.widgets.get("catalog")
ingestion_id = dbutils.widgets.get("ingestion_id")
with TaskExecution(spark=spark, catalog=catalog, task_key=TASK_KEY, task_run_id=dbutils.widgets.get("task_run_id") or None, ingestion_id=ingestion_id) as task_execution:
    result = finalize_ingestion_step(spark=spark, ingestion_id=ingestion_id, catalog=catalog)
    task_execution.complete_from_result(result)
print(json.dumps(result, sort_keys=True))
